# Inspeção dos Dados Processados

## 1. Configuração

In [1]:
import polars as pl
from pathlib import Path

# Caminhos principais do projeto
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
RAW = ROOT / "data" / "raw"
INDICADORES = ROOT / "data" / "processed" / "indicadores"
ENEM = ROOT / "data" / "processed" / "enem"

pl.Config.set_tbl_cols(20)

polars.config.Config

## 2. Inventário de Arquivos Processados

In [2]:
# Lista todos os Parquets em data/processed/
for arquivo in sorted((ROOT / "data" / "processed").rglob("*.parquet")):
    rel = arquivo.relative_to(ROOT / "data" / "processed")
    df_scan = pl.scan_parquet(arquivo)
    n_cols = len(df_scan.collect_schema())
    n_rows = df_scan.select(pl.len()).collect().item()
    print(f"{str(rel):<52} {n_rows:>9,} linhas | {n_cols:>3} cols | {arquivo.stat().st_size / 1e6:5.1f} MB")

enem/enem_escolas_2024.parquet                          29,400 linhas |  45 cols |   2.6 MB
enem/enem_itens_desempenho_2024.parquet                  4,320 linhas |  28 cols |   0.2 MB
enem/itens_cn_2024.parquet                               1,350 linhas |  19 cols |   0.0 MB
indicadores/data_adequacao_uf.parquet                       29 linhas |   4 cols |   0.0 MB
indicadores/data_afd.parquet                           176,267 linhas |  44 cols |   4.9 MB
indicadores/data_atu.parquet                           172,871 linhas |  31 cols |   4.3 MB
indicadores/data_dsu.parquet                           178,997 linhas |  19 cols |   3.0 MB
indicadores/data_inse.parquet                           69,756 linhas |  22 cols |   2.3 MB
indicadores/data_tdi.parquet                           127,929 linhas |  26 cols |   3.2 MB
indicadores/indicadores_ensino_medio_2024.parquet      176,267 linhas |  41 cols |   4.9 MB


## 3. Validação do Cruzamento (Indicadores x ENEM por Escola)

### 3.1 União por CO_ENTIDADE == CO_ESCOLA

In [3]:
indicadores = pl.read_parquet(INDICADORES / "indicadores_ensino_medio_2024.parquet")
enem = pl.read_parquet(ENEM / "enem_escolas_2024.parquet")

base_tcc = indicadores.join(enem, left_on="CO_ENTIDADE", right_on="CO_ESCOLA", how="inner")
print(f"Escolas pareadas (Censo + ENEM): {base_tcc.height:,} | Total de colunas: {base_tcc.width}")
base_tcc.head(5)

Escolas pareadas (Censo + ENEM): 25,144 | Total de colunas: 85


NU_ANO_CENSO,NO_REGIAO,SG_UF,CO_MUNICIPIO,NO_MUNICIPIO,CO_ENTIDADE,NO_ENTIDADE,NO_CATEGORIA,NO_DEPENDENCIA,AFD_MED_CAT_1,…,MEDIA_COMP1,MEDIA_COMP2,MEDIA_COMP3,MEDIA_COMP4,MEDIA_COMP5,MEDIA_GERAL,MEDIANA_GERAL,STD_GERAL,IN_MIN_10_ALUNOS,NO_MUNICIPIO_ESC
i64,str,str,i64,str,i64,str,str,str,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,bool,str
2024,"""Norte""","""RO""",1100015,"""Alta Floresta D'Oeste""",11024682,"""EEEFM EURIDICE LOPES PEDROSO""","""Urbana""","""Estadual""",86.2,…,101.51,123.77,98.49,116.23,104.91,482.93,478.24,73.22,true,"""Alta Floresta D'Oeste"""
2024,"""Norte""","""RO""",1100015,"""Alta Floresta D'Oeste""",11024968,"""EEEMTI JUSCELINO KUBITSCHEK DE…","""Urbana""","""Estadual""",79.8,…,124.21,153.16,122.63,142.63,150.0,556.37,560.04,76.91,true,"""Alta Floresta D'Oeste"""
2024,"""Norte""","""RO""",1100015,"""Alta Floresta D'Oeste""",11025638,"""COLEGIO TIRADENTES DA POLICIA …","""Urbana""","""Estadual""",73.0,…,119.2,146.0,115.2,131.6,138.8,508.95,503.2,72.97,true,"""Alta Floresta D'Oeste"""
2024,"""Norte""","""RO""",1100023,"""Ariquemes""",11006757,"""CENTRO ESTADUAL DE EDUCACAO DE…","""Urbana""","""Estadual""",null,…,95.0,107.5,92.5,100.0,80.0,465.09,461.04,57.18,false,"""Ariquemes"""
2024,"""Norte""","""RO""",1100023,"""Ariquemes""",11006773,"""EEEFM CORA CORALINA""","""Urbana""","""Estadual""",60.5,…,99.51,126.83,96.59,99.51,85.37,469.51,467.06,70.71,true,"""Ariquemes"""


### 3.2 Panorama de Física por Rede de Ensino

In [4]:
# Compara adequação docente, INSE, tamanho de turma e desempenho em Física por rede
resumo_rede = (
    base_tcc.filter(pl.col("IN_MIN_10_ALUNOS"))
    .group_by("NO_DEPENDENCIA")
    .agg([
        pl.len().alias("ESCOLAS"),
        pl.col("AFD_MED_CAT_1").mean().round(1).alias("AFD_G1_%"),
        pl.col("MEDIA_INSE").mean().round(2).alias("INSE_MEDIO"),
        pl.col("ATU_MED_TOTAL").mean().round(1).alias("ALUNOS_TURMA"),
        pl.col("PCT_ACERTO_FISICA").mean().round(1).alias("ACERTO_FISICA_%"),
        pl.col("PCT_ACERTO_BIOLOGIA").mean().round(1).alias("ACERTO_BIO_%"),
        pl.col("PCT_ACERTO_QUIMICA").mean().round(1).alias("ACERTO_QUI_%"),
        pl.col("MEDIA_CN").mean().round(1).alias("NOTA_TRI_CN"),
    ])
    .sort("ACERTO_FISICA_%", descending=True)
)
display(resumo_rede)

NO_DEPENDENCIA,ESCOLAS,AFD_G1_%,INSE_MEDIO,ALUNOS_TURMA,ACERTO_FISICA_%,ACERTO_BIO_%,ACERTO_QUI_%,NOTA_TRI_CN
str,u32,f64,f64,f64,f64,f64,f64,f64
"""Privada""",6469,70.4,null,27.8,37.1,37.9,36.7,540.5
"""Federal""",596,81.2,5.25,31.4,35.1,34.9,34.7,526.6
"""Municipal""",92,73.4,5.36,27.6,28.5,29.6,29.5,488.8
"""Estadual""",15842,68.7,4.88,29.1,24.7,26.4,27.2,464.3
